## Hücre 1 — Google Drive Bağlantısı

Google Drive'ı bağlayarak veri setlerine ve çıktı dizinlerine erişim sağlıyoruz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Hücre 2 — Dizin Yolları (Paths)

MVTec AD veri seti ve çıktı dizinlerinin yollarını tanımlıyoruz.

In [ ]:
import os

# Dizin yolları
MVTEC_ROOT = "/content/drive/MyDrive/datasets/mvtec_ad"  # gerekirse değiştir
OUT_ROOT   = "/content/drive/MyDrive/experiments/clip_dinov2_hybrid_clean"

# Çıktı dizinini oluştur
os.makedirs(OUT_ROOT, exist_ok=True)

# Dizin kontrolü
if not os.path.exists(MVTEC_ROOT):
    print("⚠️ UYARI: MVTEC_ROOT dizini bulunamadı!")
    print("Lütfen MVTEC_ROOT yolunu kontrol edin.")
else:
    print("✅ MVTEC_ROOT bulundu:", MVTEC_ROOT)
print("✅ OUT_ROOT:", OUT_ROOT)

## Hücre 3 — Kütüphane Kurulumu

Gerekli tüm Python kütüphanelerini kuruyoruz:
- **pandas**: Veri analizi için
- **ftfy, regex**: Metin işleme
- **opencv-python**: Görüntü işleme
- **scikit-learn**: Makine öğrenmesi metrikleri
- **CLIP**: OpenAI'nin CLIP modeli

In [ ]:
!pip -q install ftfy regex tqdm opencv-python scikit-learn pillow
!pip -q install git+https://github.com/openai/CLIP.git

## Hücre 4 — Kütüphane İmportları ve Cihaz Ayarları

Gerekli tüm kütüphaneleri import ediyoruz ve GPU/CPU cihaz ayarlarını yapıyoruz.

In [ ]:
import glob
import json
import time

# Reproducibility (Tekrarlanabilirlik) Sabitlemesi
import random
import numpy as np
import torch
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import gc
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm import tqdm

import torch
from sklearn.metrics import roc_auc_score

# Cihaz ayarları
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Kullanılan cihaz: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Belleği: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Hücre 5 — MVTec Yardımcı Fonksiyonları

MVTec AD veri setini okumak ve işlemek için gerekli yardımcı fonksiyonlar:
- Veri seti kategorilerini listeleme
- Görüntü ve maske yükleme
- Normalizasyon işlemleri
- Path (yol) yönetimi

In [ ]:
def list_mvtec_categories(root):
    """
    MVTec AD kategorilerini listele.
    
    Args:
        root: MVTec AD kök dizini
    Returns:
        Sıralanmış kategori listesi
    """
    if not os.path.exists(root):
        raise ValueError(f"Root dizini bulunamadı: {root}")
    
    cats = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    cats.sort()
    return cats

def load_image_rgb(path, resize=None):
    """
    Görüntüyü RGB formatında yükle.
    
    Args:
        path: Görüntü dosya yolu
        resize: Yeniden boyutlandırma boyutu (None ise orijinal boyut)
    Returns:
        PIL Image objesi
    """
    try:
        img = Image.open(path).convert("RGB")
        if resize is not None:
            img = img.resize((resize, resize), Image.BICUBIC)
        return img
    except Exception as e:
        print(f"⚠️ Görüntü yükleme hatası {path}: {e}")
        return None

def load_mask(path, resize=None):
    """
    Ground truth maske yükle.
    
    Args:
        path: Maske dosya yolu
        resize: Yeniden boyutlandırma boyutu
    Returns:
        Binary numpy array (0 veya 1)
    """
    try:
        m = Image.open(path).convert("L")
        if resize is not None:
            m = m.resize((resize, resize), Image.NEAREST)
        m = np.array(m, dtype=np.uint8)
        m = (m > 127).astype(np.uint8)
        return m
    except Exception as e:
        print(f"⚠️ Maske yükleme hatası {path}: {e}")
        return None

def normalize_scores(x):
    """
    Skorları [0, 1] aralığına normalize et.
    
    Args:
        x: Numpy array
    Returns:
        Normalize edilmiş array
    """
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return x
    mn, mx = x.min(), x.max()
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

def l2_normalize(x, axis=1, eps=1e-12):
    """
    L2 normalizasyon uygula.
    
    Args:
        x: Numpy array
        axis: Normalizasyon ekseni
        eps: Sıfıra bölmeyi önlemek için küçük değer
    Returns:
        L2 normalize edilmiş array
    """
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / (n + eps)

def _basename_noext(p):
    """Dosya adını uzantısız al."""
    return os.path.splitext(os.path.basename(p))[0]

def get_mvtec_paths(root, category):
    """
    MVTec AD kategorisi için dosya yollarını al.
    
    Args:
        root: MVTec AD kök dizini
        category: Kategori adı (örn: 'capsule', 'bottle')
        
    Returns:
        train_good: Eğitim görüntüleri listesi
        test_items: Test görüntüleri [(img_path, label, mask_path_or_None)]
                   label: 0=normal, 1=anomali
    """
    cat_root = os.path.join(root, category)
    if not os.path.exists(cat_root):
        raise ValueError(f"Kategori dizini bulunamadı: {cat_root}")
    
    train_good = sorted(glob.glob(os.path.join(cat_root, "train", "good", "*.*")))
    
    if len(train_good) == 0:
        print(f"⚠️ UYARI: '{category}' için eğitim görüntüsü bulunamadı!")

    test_dir = os.path.join(cat_root, "test")
    gt_dir   = os.path.join(cat_root, "ground_truth")

    test_items = []
    if not os.path.exists(test_dir):
        print(f"⚠️ UYARI: Test dizini bulunamadı: {test_dir}")
        return train_good, test_items
    
    defect_types = [d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))]
    defect_types.sort()

    for d in defect_types:
        img_paths = sorted(glob.glob(os.path.join(test_dir, d, "*.*")))
        if d == "good":
            for p in img_paths:
                test_items.append((p, 0, None))
            continue

        gt_subdir = os.path.join(gt_dir, d)
        mask_paths = sorted(glob.glob(os.path.join(gt_subdir, "*.*"))) if os.path.exists(gt_subdir) else []

        mask_map = {_basename_noext(m): m for m in mask_paths}
        for m in mask_paths:
            b = _basename_noext(m)
            if b.endswith("_mask"):
                mask_map[b[:-5]] = m

        if len(mask_paths) == len(img_paths) and len(img_paths) > 0:
            fallback_pairs = dict(zip([_basename_noext(p) for p in img_paths], mask_paths))
        else:
            fallback_pairs = {}

        for p in img_paths:
            b = _basename_noext(p)
            mpath = mask_map.get(b, None)
            if mpath is None:
                mpath = fallback_pairs.get(b, None)
            test_items.append((p, 1, mpath))

    return train_good, test_items

## Hücre 6 — CLIP Model Yükleme ve İşlevler

OpenAI'nin CLIP modelini kullanarak:
- Model yükleme ve inicializasyon
- Görüntü embedding çıkarma
- Anomali skoru hesaplama
- Referans feature oluşturma
- Threshold (eşik) belirleme

In [ ]:
import clip

# CLIP modelini yükle
print("📦 CLIP modeli yükleniyor...")
clip_model, clip_preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()
print("✅ CLIP modeli başarıyla yüklendi!")

@torch.no_grad()
def clip_embed_pil(pil_img):
    """
    PIL görüntüsünden CLIP embedding çıkar.
    
    Args:
        pil_img: PIL Image objesi
    Returns:
        Normalize edilmiş feature vektörü (D,)
    """
    if pil_img is None:
        raise ValueError("Geçersiz görüntü")
    
    x = clip_preprocess(pil_img).unsqueeze(0).to(device)
    feat = clip_model.encode_image(x).float()
    feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.squeeze(0)  # (D,)

@torch.no_grad()
def clip_anomaly_score(pil_img, clip_ref):
    """
    CLIP kullanarak anomali skoru hesapla.
    
    Args:
        pil_img: Test görüntüsü
        clip_ref: Referans feature vektörü
    Returns:
        Anomali skoru (0-1 arası, yüksek = daha anormal)
    """
    f = clip_embed_pil(pil_img).cpu().numpy().astype(np.float32)
    sim = float(np.dot(f, clip_ref) / (np.linalg.norm(f) * np.linalg.norm(clip_ref) + 1e-12))
    return 1.0 - sim

@torch.no_grad()
def build_clip_reference(train_good_paths, max_images=50):
    """
    Eğitim görüntülerinden CLIP referans vektörü oluştur.
    
    Args:
        train_good_paths: Normal eğitim görüntüleri
        max_images: Maksimum görüntü sayısı
    Returns:
        Ortalama normalize edilmiş referans vektörü
    """
    feats = []
    paths = train_good_paths[:max_images]
    
    for p in tqdm(paths, desc="🔍 CLIP referans", unit="img"):
        img = load_image_rgb(p, resize=None)
        if img is None:
            continue
        f = clip_embed_pil(img).cpu().numpy()
        feats.append(f)
    
    if len(feats) == 0:
        raise ValueError("Hiçbir geçerli görüntü yüklenemedi!")
    
    feats = np.stack(feats, axis=0)
    ref = feats.mean(axis=0)
    ref = ref / (np.linalg.norm(ref) + 1e-12)
    
    # Bellek temizliği
    del feats
    gc.collect()
    
    return ref.astype(np.float32)

@torch.no_grad()
def estimate_clip_threshold(train_good_paths, clip_ref, max_images=50, quantile=0.95):
    """
    Normal görüntüler üzerinden CLIP threshold tahmin et.
    
    Args:
        train_good_paths: Normal eğitim görüntüleri
        clip_ref: Referans vektörü
        max_images: Maksimum görüntü sayısı
        quantile: Quantile değeri (0.95 = %95'lik dilim)
    Returns:
        Threshold değeri
    """
    scores = []
    paths = train_good_paths[:max_images]
    
    for p in tqdm(paths, desc="🎯 CLIP threshold", unit="img"):
        img = load_image_rgb(p, resize=None)
        if img is None:
            continue
        s = clip_anomaly_score(img, clip_ref)
        scores.append(s)
    
    if len(scores) == 0:
        raise ValueError("Threshold hesaplanamadı: Geçerli görüntü yok")
    
    thr = float(np.quantile(np.array(scores, dtype=np.float32), quantile))
    return thr

## Hücre 7 — DINOv2 Model ve Anomali Haritası

Facebook'un DINOv2 modelini kullanarak:
- Patch-level feature çıkarma
- Normal görüntülerden patch bank oluşturma
- Coreset subsampling
- k-NN (k-Nearest Neighbors) indexleme
- Pixel-level anomali haritası üretme

In [ ]:
import torchvision.transforms as T
from sklearn.neighbors import NearestNeighbors

# DINOv2 modelini yükle
print("📦 DINOv2 modeli yükleniyor...")
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
dinov2 = dinov2.to(device)
dinov2.eval()
print("✅ DINOv2 modeli başarıyla yüklendi!")

# Sabit değerler
DINO_SIZE = 518  # vitb14 için tipik girdi boyutu

# DINOv2 preprocessing transform
dino_tf = T.Compose([
    T.Resize((DINO_SIZE, DINO_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

@torch.no_grad()
def dinov2_patch_features(pil_img):
    """
    DINOv2'den patch-level feature'lar çıkar.
    
    Args:
        pil_img: PIL Image objesi
    Returns:
        patch_features: (N, D) feature tensor
        grid_size: (height, width) patch grid boyutu
    """
    if pil_img is None:
        raise ValueError("Geçersiz görüntü")
    
    x = dino_tf(pil_img).unsqueeze(0).to(device)
    feats = dinov2.forward_features(x)
    p = feats["x_norm_patchtokens"][0]  # (N, D)
    n = p.shape[0]
    g = int(np.sqrt(n))
    return p, (g, g)

@torch.no_grad()
def build_dino_patch_bank(train_good_paths, max_images=30, max_patches_per_image=400):
    """
    Normal görüntülerden DINOv2 patch bank oluştur.
    
    Args:
        train_good_paths: Normal eğitim görüntüleri
        max_images: Maksimum görüntü sayısı
        max_patches_per_image: Görüntü başına max patch sayısı
    Returns:
        L2 normalize edilmiş patch feature array (N_total, D)
    """
    bank = []
    paths = train_good_paths[:max_images]
    
    for p in tqdm(paths, desc="🧩 DINOv2 patch bank", unit="img"):
        img = load_image_rgb(p, resize=None)
        if img is None:
            continue
        
        pf, _ = dinov2_patch_features(img)
        pf = pf.detach().cpu().numpy().astype(np.float32)

        if pf.shape[0] > max_patches_per_image:
            idx = np.random.choice(pf.shape[0], size=max_patches_per_image, replace=False)
            pf = pf[idx]
        bank.append(pf)
    
    if len(bank) == 0:
        raise ValueError("Patch bank oluşturulamadı: Geçerli görüntü yok")

    bank = np.concatenate(bank, axis=0)
    bank = l2_normalize(bank, axis=1)
    
    # GPU belleği temizle
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    print(f"   📊 Patch bank boyutu: {bank.shape}")
    return bank.astype(np.float32)

def coreset_subsample(bank, target_size=12000, seed=0):
    """
    Patch bank'i coreset subsampling ile küçült.
    
    Args:
        bank: Patch feature array
        target_size: Hedef boyut (None ise küçültme yapılmaz)
        seed: Random seed
    Returns:
        Subsample edilmiş patch array
    """
    rng = np.random.default_rng(seed)
    n = bank.shape[0]
    
    if target_size is None or target_size >= n:
        print(f"   ℹ️  Coreset subsampling atlandı (bank size: {n})")
        return bank
    
    idx = rng.choice(n, size=target_size, replace=False)
    print(f"   ✂️  Coreset: {n} -> {target_size} patches")
    return bank[idx]

def build_knn_index(bank_coreset, k=5):
    """
    k-NN index oluştur.
    
    Args:
        bank_coreset: Referans patch features
        k: Komşu sayısı
    Returns:
        Sklearn NearestNeighbors objesi
    """
    print(f"   🔍 k-NN index oluşturuluyor (k={k})...")
    nn = NearestNeighbors(n_neighbors=k, algorithm="auto", metric="cosine")
    nn.fit(bank_coreset)
    print("   ✅ k-NN index hazır")
    return nn

@torch.no_grad()
def dino_anomaly_map_knn(pil_img, nn_index, k=5, out_size=256):
    """
    DINOv2 + k-NN kullanarak pixel-level anomali haritası üret.
    
    Args:
        pil_img: Test görüntüsü
        nn_index: k-NN index
        k: Komşu sayısı
        out_size: Çıktı harita boyutu
    Returns:
        Normalize edilmiş anomali haritası (out_size x out_size)
    """
    pf, (gh, gw) = dinov2_patch_features(pil_img)
    pf = pf.detach().cpu().numpy().astype(np.float32)
    pf = l2_normalize(pf, axis=1)

    dists, _ = nn_index.kneighbors(pf, n_neighbors=k, return_distance=True)
    score = dists.mean(axis=1)  # (N,)
    amap = score.reshape(gh, gw)
    amap = normalize_scores(amap)

    amap_up = cv2.resize(amap, (out_size, out_size), interpolation=cv2.INTER_CUBIC)
    return amap_up

## Hücre 8 — Üç Farklı Yöntem İmplementasyonu

### Üç farklı anomali tespit yaklaşımı:

1. **Hybrid (Hibrit)**: CLIP (görüntü-level filtreleme) + DINOv2 (pixel-level lokalizasyon)
   - CLIP ile hızlı ön tarama
   - Flagged görüntülerde DINOv2 ile detaylı analiz
   
2. **CLIP-only**: Sadece CLIP ile görüntü-level anomali tespiti
   - Hızlı ve hafif
   - Lokalizasyon yok
   
3. **DINO-only**: Sadece DINOv2 ile pixel-level anomali tespiti
   - En detaylı analiz
   - En yüksek hesaplama maliyeti

In [ ]:
def image_score_from_map(amap_flat: np.ndarray, method="max", topk=200) -> float:
    a = amap_flat.astype(np.float32).ravel()
    if a.size == 0:
        return 0.0
    if method == "max":
        return float(np.max(a))
    if method == "topk":
        k = min(int(topk), a.size)
        idx = np.argpartition(a, -k)[-k:]
        return float(np.mean(a[idx]))
    raise ValueError("method must be 'max' or 'topk'")

def run_category_hybrid(category,
                        clip_ref_max=50,
                        clip_thr_quantile=0.95,
                        dino_ref_imgs=30,
                        dino_patches_per_img=400,
                        dino_coreset=12000,
                        knn_k=5,
                        eval_size=256):

    train_good, test_items = get_mvtec_paths(MVTEC_ROOT, category)

    clip_ref = build_clip_reference(train_good, max_images=clip_ref_max)
    clip_thr = estimate_clip_threshold(train_good, clip_ref,
                                       max_images=clip_ref_max,
                                       quantile=clip_thr_quantile)

    patch_bank = build_dino_patch_bank(train_good,
                                       max_images=dino_ref_imgs,
                                       max_patches_per_image=dino_patches_per_img)
    bank_cs = coreset_subsample(patch_bank, target_size=dino_coreset, seed=0)
    nn_index = build_knn_index(bank_cs, k=knn_k)

    y_img, s_img = [], []
    y_pix_cond, s_pix_cond = [], []
    y_pix_e2e,  s_pix_e2e  = [], []

    flagged_count = 0
    masked_count  = 0

    for (img_path, label, mask_path) in tqdm(test_items, desc=f"Test HYBRID {category}"):
        img = load_image_rgb(img_path, resize=None)

        s = clip_anomaly_score(img, clip_ref)
        y_img.append(label)
        s_img.append(s)

        has_gt = (mask_path is not None and os.path.exists(mask_path))
        if not has_gt:
            continue

        masked_count += 1
        gt = load_mask(mask_path, resize=eval_size).reshape(-1)

        flagged = (s > clip_thr)
        if flagged:
            flagged_count += 1
            amap = dino_anomaly_map_knn(img, nn_index, k=knn_k, out_size=eval_size).reshape(-1)
            y_pix_cond.append(gt); s_pix_cond.append(amap)
            y_pix_e2e.append(gt);  s_pix_e2e.append(amap)
        else:
            zeros = np.zeros_like(gt, dtype=np.float32)
            y_pix_e2e.append(gt); s_pix_e2e.append(zeros)

    auroc_img = roc_auc_score(y_img, s_img) if len(np.unique(y_img)) > 1 else float("nan")

    if len(y_pix_cond) > 0:
        ypc = np.concatenate(y_pix_cond, axis=0)
        spc = np.concatenate(s_pix_cond, axis=0)
        auroc_pix_cond = roc_auc_score(ypc, spc) if len(np.unique(ypc)) > 1 else float("nan")
    else:
        auroc_pix_cond = float("nan")

    if len(y_pix_e2e) > 0:
        ype = np.concatenate(y_pix_e2e, axis=0)
        spe = np.concatenate(s_pix_e2e, axis=0)
        auroc_pix_e2e = roc_auc_score(ype, spe) if len(np.unique(ype)) > 1 else float("nan")
    else:
        auroc_pix_e2e = float("nan")

    flag_rate = (flagged_count / masked_count) if masked_count > 0 else float("nan")

    return {
        "category": category,
        "auroc_img": auroc_img,
        "auroc_pix_cond": auroc_pix_cond,
        "auroc_pix_e2e": auroc_pix_e2e,
        "clip_thr": float(clip_thr),
        "flag_rate": float(flag_rate),
        "n_masked": int(masked_count),
    }

def run_category_clip_only(category, clip_ref_max=50):
    train_good, test_items = get_mvtec_paths(MVTEC_ROOT, category)
    clip_ref = build_clip_reference(train_good, max_images=clip_ref_max)

    y_img, s_img = [], []
    for (img_path, label, mask_path) in tqdm(test_items, desc=f"Test CLIP {category}"):
        img = load_image_rgb(img_path, resize=None)
        s = clip_anomaly_score(img, clip_ref)
        y_img.append(label)
        s_img.append(s)

    auroc_img = roc_auc_score(y_img, s_img) if len(np.unique(y_img)) > 1 else float("nan")
    return {"category": category, "auroc_img": auroc_img}

def run_category_dino_only(category,
                           dino_ref_imgs=30,
                           dino_patches_per_img=400,
                           dino_coreset=12000,
                           knn_k=5,
                           eval_size=256,
                           img_score_method="max",
                           img_score_topk=200):

    train_good, test_items = get_mvtec_paths(MVTEC_ROOT, category)

    patch_bank = build_dino_patch_bank(train_good,
                                       max_images=dino_ref_imgs,
                                       max_patches_per_image=dino_patches_per_img)
    bank_cs = coreset_subsample(patch_bank, target_size=dino_coreset, seed=0)
    nn_index = build_knn_index(bank_cs, k=knn_k)

    y_img, s_img = [], []
    y_pix, s_pix = [], []
    masked_count = 0

    for (img_path, label, mask_path) in tqdm(test_items, desc=f"Test DINO {category}"):
        img = load_image_rgb(img_path, resize=None)

        amap = dino_anomaly_map_knn(img, nn_index, k=knn_k, out_size=eval_size).reshape(-1)
        s_img.append(image_score_from_map(amap, method=img_score_method, topk=img_score_topk))
        y_img.append(label)

        has_gt = (mask_path is not None and os.path.exists(mask_path))
        if not has_gt:
            continue

        masked_count += 1
        gt = load_mask(mask_path, resize=eval_size).reshape(-1)

        y_pix.append(gt)
        s_pix.append(amap.astype(np.float32))

    auroc_img = roc_auc_score(y_img, s_img) if len(np.unique(y_img)) > 1 else float("nan")

    if len(y_pix) > 0:
        yp = np.concatenate(y_pix, axis=0)
        sp = np.concatenate(s_pix, axis=0)
        auroc_pix_e2e = roc_auc_score(yp, sp) if len(np.unique(yp)) > 1 else float("nan")
    else:
        auroc_pix_e2e = float("nan")

    return {
        "category": category,
        "auroc_img": auroc_img,
        "auroc_pix_e2e": auroc_pix_e2e,
        "n_masked": int(masked_count),
        "img_score_method": img_score_method,
        "img_score_topk": int(img_score_topk),
    }

## Threshold Ablation (Eşik Değeri Seçimi)

Hybrid yöntemde CLIP skorunun % kaçlık dilimini (quantile) eşik olarak alacağımızı belirlemek için minik bir ablation çalışması:
- Neden `0.93` seçildiğini doğrulamak için farklı oranlar (%90, %93, %95, %97, %99) test edilmiştir.
- Neden `0.93` kullanıldığına dair argüman oluşturmak için `mean_pix_e2e` ve `mean_img` değerleri arasındaki dengeye bakılır.

In [ ]:
import numpy as np
import pandas as pd
import os

def sweep_threshold_quantiles(quantiles=(0.90, 0.93, 0.95, 0.97, 0.99), out_dir=OUT_ROOT):
    rows = []
    for q in quantiles:
        vals_img, vals_pc, vals_pe, vals_fr = [], [], [], []
        for c in list_mvtec_categories(MVTEC_ROOT):
            r = run_category_hybrid(c, clip_thr_quantile=q)
            vals_img.append(r["auroc_img"])
            vals_pc.append(r["auroc_pix_cond"])
            vals_pe.append(r["auroc_pix_e2e"])
            vals_fr.append(r.get("flag_rate", np.nan))
        row = {
            "quantile": q,
            "mean_img": float(np.nanmean(vals_img)),
            "mean_pix_cond": float(np.nanmean(vals_pc)),
            "mean_pix_e2e": float(np.nanmean(vals_pe)),
            "mean_flag_rate": float(np.nanmean(vals_fr)),
        }
        print("q=", q, row)
        rows.append(row)

    dfq = pd.DataFrame(rows)
    dfq.to_csv(os.path.join(out_dir, "Ablation_threshold_quantiles.csv"), index=False)
    return dfq

# Not: Bu blok uzun sürebilir, tüm MVTec üzerinde threshold taraması yapar.
# dfq = sweep_threshold_quantiles()
# dfq


## Hücre 9 — Değerlendirme Runner (CSV/JSON Kayıt)

Tüm kategoriler üzerinde seçilen yöntemi çalıştırır ve sonuçları kaydeder:
- Her kategori için metrikler hesaplanır (AUROC)
- Sonuçlar CSV formatında kaydedilir
- Özet istatistikler JSON formatında kaydedilir
- Çalışma süreleri raporlanır

In [ ]:
def eval_all(method_name: str, run_fn, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)

    cats = list_mvtec_categories(MVTEC_ROOT)
    rows = []
    t0 = time.time()

    for c in cats:
        t1 = time.time()
        r = run_fn(c)
        rows.append({**{"category": c, "method": method_name, "runtime_s": time.time()-t1}, **r})

        show = []
        for k in ["auroc_img","auroc_pix_cond","auroc_pix_e2e","flag_rate"]:
            if k in r and isinstance(r[k], (int,float)) and r[k]==r[k]:
                show.append(f"{k}={r[k]:.4f}")
        print(c, " | ".join(show))

    df = np.nan_to_num(pd.DataFrame(rows), nan=np.nan)
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out_dir, f"results_{method_name}.csv"), index=False)

    summary = {"method": method_name, "total_runtime_s": float(time.time()-t0)}
    for col in ["auroc_img","auroc_pix_cond","auroc_pix_e2e","flag_rate","runtime_s"]:
        if col in df.columns:
            summary[f"mean_{col}"] = float(np.nanmean(df[col].values))

    with open(os.path.join(out_dir, f"summary_{method_name}.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("Saved:", out_dir)
    return df, summary

## Qualitative Figures: Amaç ve Mantık

### Qualitative Panel Üretimi (Makale Figürleri)

Bu bölüm, makalede kullanılabilecek 3-panelli görseller üretir:

**Panel 1**: Orijinal görüntü + CLIP skoru + flagged bilgisi  
**Panel 2**: Ground-truth maske (MVTec ground_truth)  
**Panel 3**: DINOv2 heatmap overlay (anomali lokalizasyonu)

### Neden Önemli?

Hibrit sistemde CLIP image-level filtre görevi görür (flagged True/False).

Ancak görsel analiz için bazen **flagged=False** olsa bile DINO'nun ne "gördüğünü" göstermek isteriz.

### İki Mod:

- **force_dino=True**: CLIP gating'e bakmadan DINO heatmap üret (analiz/figür için)
- **force_dino=False**: Gerçek pipeline davranışı (sadece flagged ise DINO çalışır)

### Otomatik Örnek Seçimi:

"Hikâye anlatımı" için dengeli seçim:
- **n_flagged_true** adet flagged=True örnek
- **n_flagged_false** adet flagged=False örnek

Böylece figürde hem "gating çalıştı" hem "gating kaçırdı / conservative kaldı" durumları birlikte gösterilir.

In [ ]:
df_h, sum_h = eval_all(
    "hybrid",
    lambda c: run_category_hybrid(c, clip_thr_quantile=0.93),
    out_dir=os.path.join(OUT_ROOT, "01_hybrid_q0p93")
)

df_c, sum_c = eval_all(
    "clip_only",
    lambda c: run_category_clip_only(c),
    out_dir=os.path.join(OUT_ROOT, "02_clip_only")
)

df_d, sum_d = eval_all(
    "dino_only",
    lambda c: run_category_dino_only(c, img_score_method="max"),
    out_dir=os.path.join(OUT_ROOT, "03_dino_only_max")
)

sum_h, sum_c, sum_d

## Hücre 11 — Karşılaştırmalı Excel Raporu Oluştur

Tüm yöntemlerin sonuçlarını karşılaştıran Excel dosyası üretir:
- **Sheet 1**: Image-level AUROC karşılaştırması
- **Sheet 2**: Pixel-level end-to-end AUROC
- **Sheet 3**: Pixel-level conditional AUROC (sadece flagged)
- **Sheet 4**: Ham veriler (tüm metrikler)

Excel dosyası pivot tablolar kullanarak kolay karşılaştırma sağlar.

In [ ]:
import pandas as pd

def export_comparison(df_h, df_c, df_d, out_path):
    all_df = pd.concat([df_h, df_c, df_d], ignore_index=True)

    piv_img = all_df.pivot_table(index="category", columns="method", values="auroc_img", aggfunc="mean")

    piv_pix_e2e = all_df.pivot_table(index="category", columns="method",
                                     values="auroc_pix_e2e", aggfunc="mean") if "auroc_pix_e2e" in all_df.columns else None
    piv_pix_cond = all_df.pivot_table(index="category", columns="method",
                                      values="auroc_pix_cond", aggfunc="mean") if "auroc_pix_cond" in all_df.columns else None

    with pd.ExcelWriter(out_path) as w:
        piv_img.to_excel(w, sheet_name="image_auroc")
        if piv_pix_e2e is not None:
            piv_pix_e2e.to_excel(w, sheet_name="pixel_e2e_auroc")
        if piv_pix_cond is not None:
            piv_pix_cond.to_excel(w, sheet_name="pixel_cond_auroc")
        all_df.to_excel(w, sheet_name="raw", index=False)

    print("Saved:", out_path)
    return piv_img

piv = export_comparison(df_h, df_c, df_d, os.path.join(OUT_ROOT, "comparison.xlsx"))
piv

## Hücre 12 — Hızlı Doğrulama (Sanity Check)

Tüm yöntemlerin ortalama metriklerini ekrana yazdırarak hızlı bir doğrulama yapar:
- Hybrid yöntem için image ve pixel AUROC değerleri
- CLIP-only için image AUROC
- DINO-only için image ve pixel AUROC değerleri
- Flag rate (CLIP'in kaç görüntüyü DINOv2'ye yönlendirdiği)

In [ ]:
print("HYBRID mean img:", np.nanmean(df_h["auroc_img"]))
print("HYBRID mean pix_e2e:", np.nanmean(df_h["auroc_pix_e2e"]))
print("HYBRID mean flag_rate:", np.nanmean(df_h.get("flag_rate", np.nan)))

print("CLIP_ONLY mean img:", np.nanmean(df_c["auroc_img"]))

print("DINO_ONLY mean img:", np.nanmean(df_d["auroc_img"]))
print("DINO_ONLY mean pix_e2e:", np.nanmean(df_d["auroc_pix_e2e"]))

##Qualitative Figures: Amaç ve Mantık

##Qualitative Panel Üretimi (Makale Figürleri)

Bu bölüm, makalede kullanılabilecek 3-panelli görseller üretir:

Orijinal görüntü + CLIP skoru + flagged bilgisi

Ground-truth maske (MVTec ground_truth)

DINOv2 heatmap overlay (anomali lokalizasyonu)

##Neden önemli?

Hibrit sistemde CLIP image-level filtre görevi görür (flagged True/False).

Ancak görsel analiz için bazen flagged=False olsa bile DINO’nun ne “gördüğünü” göstermek isteriz.

Bu yüzden iki mod vardır:

force_dino=True: CLIP gating’e bakmadan DINO heatmap üret (analiz/figür için)

force_dino=False: gerçek pipeline davranışı (sadece flagged ise DINO çalışır)

Ek olarak “hikâye anlatımı” için otomatik seçim yapıyoruz:

n_flagged_true adet flagged=True örnek

n_flagged_false adet flagged=False örnek
Böylece figürde hem “gating çalıştı” hem “gating kaçırdı / conservative kaldı” durumları birlikte gösterilir.

In [ ]:
# ================================================================
# Qualitative Figure Utilities (Görsel Örnekler için Yardımcılar)
# ================================================================

"""
Bu hücrenin görevleri:

1) 3-panelli figür oluşturur:
   Panel 1: Görüntü + CLIP score/flag
   Panel 2: Ground truth maske
   Panel 3: DINO heatmap overlay

2) Kategoriden otomatik örnek seçer:
   - flagged=True örnekler (CLIP, DINO'yu devreye soktu)
   - flagged=False örnekler (CLIP reddetti; force_dino=True ile
     DINO'nun ne gördüğü gösterilebilir)

3) Figürleri OUT_ROOT altına PNG ve PDF olarak kaydeder

Bağımlılıklar:
- Önceki hücrelerdeki fonksiyonlar (get_mvtec_paths, load_image_rgb,
  load_mask, CLIP ve DINOv2 fonksiyonları)

Makale için kullanım:
- force_dino=True: "CLIP reddetse bile DINO ne görüyor?" analizi
- force_dino=False: Pipeline'ın gerçek davranışı
"""

import os
import random
import numpy as np
import matplotlib.pyplot as plt

def plot_qual_panel(pil_img, gt_mask, heatmap, clip_score, flagged,
                    label=None, save_path=None):
    """
    3-panelli görsel oluştur ve kaydet.
    
    Args:
        pil_img: PIL Image objesi
        gt_mask: Ground truth maske (numpy array)
        heatmap: DINOv2 anomali haritası
        clip_score: CLIP anomali skoru
        flagged: CLIP flagged durumu (True/False)
        label: Görüntü label'ı (0=normal, 1=anomali)
        save_path: Kayıt yolu (.png uzantılı ise .pdf de oluşturulur)
        
    Returns:
        matplotlib figure objesi
        
    Panel açıklaması:
      [0] Orijinal görüntü + metrikler
      [1] Ground truth maske
      [2] DINOv2 heatmap overlay
    """
    img_np = np.array(pil_img)

    fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.3))

    # Panel 1: Image + metadata
    axes[0].imshow(img_np)
    title = f"Image (CLIP={clip_score:.3f}, flagged={flagged})"
    if label is not None:
        # label: 0 normal, 1 anomaly
        title = f"Image (y={label}, CLIP={clip_score:.3f}, flagged={flagged})"
    axes[0].set_title(title)
    axes[0].axis("off")

    # Panel 2: GT mask
    if gt_mask is None:
        # gösterim için boş maske
        axes[1].imshow(np.zeros((img_np.shape[0], img_np.shape[1])), cmap="gray")
        axes[1].set_title("GT Mask (None)")
    else:
        axes[1].imshow(gt_mask, cmap="gray")
        axes[1].set_title("GT Mask")
    axes[1].axis("off")

    # Panel 3: Heatmap overlay
    axes[2].imshow(img_np)
    if heatmap is not None:
        axes[2].imshow(heatmap, cmap="jet", alpha=0.45)
    axes[2].set_title("DINOv2 Heatmap")
    axes[2].axis("off")

    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=220, bbox_inches="tight")
        print(f"   💾 Kaydedildi: {os.path.basename(save_path)}")
        
        if save_path.lower().endswith(".png"):
            pdf_path = save_path[:-4] + ".pdf"
            fig.savefig(pdf_path, bbox_inches="tight")
            print(f"   📄 PDF: {os.path.basename(pdf_path)}")
    
    # Figürü göster ve kapat (memory için)
    plt.show()
    plt.close(fig)
    
    return fig


def select_masked_anomalies(test_items):
    """
    Ground truth maske'si olan anomali görüntülerini seç.
    
    Args:
        test_items: Test görüntüleri listesi [(path, label, mask_path)]
        
    Returns:
        Mask'li anomali görüntüleri listesi
    """
    return [(p, y, m) for (p, y, m) in test_items
            if y == 1 and (m is not None and os.path.exists(m))]


def export_qual_examples_balanced(category,
                                  clip_ref_max=50,
                                  clip_thr_quantile=0.93,
                                  dino_ref_imgs=30,
                                  dino_patches_per_img=400,
                                  dino_coreset=12000,
                                  knn_k=5,
                                  eval_size=256,
                                  n_flagged_true=3,
                                  n_flagged_false=3,
                                  force_dino=True,
                                  seed=0,
                                  out_dir=None):
    """
    Kategoriden otomatik örnek seçer ve figürleri kaydeder.

    Seçim mantığı:
      - Sadece mask'i olan anomaly örnekleri arasından seçer.
      - Önce CLIP score ve flagged hesaplanır.
      - flagged=True ve flagged=False havuzlarından istenen sayıda örnek çekilir.
      - force_dino=True ise flagged=False örneklerde de DINO heatmap üretilir (analiz/figür amaçlı).
      - force_dino=False ise pipeline davranışı: flagged=False -> heatmap üretilmez (None).

    Çıktı:
      - OUT_ROOT/qual_panels/<category>/... altında PNG + PDF dosyaları
    """
    if out_dir is None:
        out_dir = os.path.join(OUT_ROOT, "qual_panels_balanced", category)

    random.seed(seed)
    np.random.seed(seed)

    # ---- paths
    train_good, test_items = get_mvtec_paths(MVTEC_ROOT, category)

    # ---- CLIP init
    clip_ref = build_clip_reference(train_good, max_images=clip_ref_max)
    clip_thr = estimate_clip_threshold(train_good, clip_ref,
                                       max_images=clip_ref_max,
                                       quantile=clip_thr_quantile)

    # ---- DINO init (tek sefer)
    patch_bank = build_dino_patch_bank(train_good,
                                       max_images=dino_ref_imgs,
                                       max_patches_per_image=dino_patches_per_img)
    bank_cs = coreset_subsample(patch_bank, target_size=dino_coreset, seed=0)
    nn_index = build_knn_index(bank_cs, k=knn_k)

    # ---- candidate pool (masked anomalies)
    candidates = select_masked_anomalies(test_items)
    if len(candidates) == 0:
        print("No masked anomalies found for:", category)
        return []

    # ---- compute CLIP score & split by flagged
    pool_true, pool_false = [], []
    for (img_path, label, mask_path) in tqdm(candidates, desc=f"Scoring candidates {category}"):
        img = load_image_rgb(img_path, resize=None)
        s = float(clip_anomaly_score(img, clip_ref))
        flagged = (s > clip_thr)
        rec = (img_path, label, mask_path, s, flagged)
        (pool_true if flagged else pool_false).append(rec)

    random.shuffle(pool_true)
    random.shuffle(pool_false)

    chosen = []
    chosen += pool_true[:n_flagged_true]
    chosen += pool_false[:n_flagged_false]

    print(f"[{category}] candidates: flagged=True {len(pool_true)}, flagged=False {len(pool_false)}")
    print(f"[{category}] chosen: flagged=True {min(n_flagged_true,len(pool_true))}, flagged=False {min(n_flagged_false,len(pool_false))}")
    print(f"force_dino={force_dino} (False ise flagged=False örnekte heatmap üretilmez)")

    saved = []
    for i, (img_path, label, mask_path, clip_score, flagged) in enumerate(chosen):
        img = load_image_rgb(img_path, resize=None)
        gt = load_mask(mask_path, resize=eval_size)

        if force_dino or flagged:
            hm = dino_anomaly_map_knn(img, nn_index, k=knn_k, out_size=eval_size)
        else:
            hm = None

        # dosya adı: T/F + score ile
        tag = "T" if flagged else "F"
        save_path = os.path.join(out_dir, f"{category}_{i:02d}_flag{tag}_clip{clip_score:.3f}.png")

        plot_qual_panel(img, gt, hm, clip_score, flagged, label=label, save_path=save_path)
        saved.append(save_path)
        print("Saved:", save_path)

    return saved

## Hücre 13 — Örnek Çalıştırma (Qualitative Figures)

### Örnek kullanım senaryoları:

**Örnek 1**: Makale figürü için (CLIP False olsa bile DINO göster)
- `force_dino=True` kullanılır
- Her durumda DINOv2 heatmap üretilir
- "CLIP kaçırdığı anomalileri DINO yakalıyor mu?" sorusunu yanıtlar

**Örnek 2**: Gerçek pipeline davranışı
- `force_dino=False` kullanılır  
- Sadece CLIP flag=True durumunda DINOv2 çalışır
- Gerçek sistem performansını gösterir

### Parametreler:
- `n_flagged_true`: CLIP'in yakaladığı örnekler
- `n_flagged_false`: CLIP'in kaçırdığı örnekler
- `clip_thr_quantile`: CLIP threshold quantile değeri

In [ ]:
# ========================================
# Örnek Çalıştırmalar
# ========================================

print("="*60)
print("📊 ÖRNEK 1: Makale figürü (force_dino=True)")
print("="*60)
# CLIP False olsa bile DINO heatmap göster (analiz için)
export_qual_examples_balanced(
    category="capsule",
    clip_thr_quantile=0.93,
    n_flagged_true=3,
    n_flagged_false=3,
    force_dino=True,
    seed=0
)

print("\n" + "="*60)
print("📊 ÖRNEK 2: Gerçek pipeline (force_dino=False)")
print("="*60)
# Gerçek davranış: flagged=False ise heatmap üretilmez
export_qual_examples_balanced(
    category="capsule",
    clip_thr_quantile=0.93,
    n_flagged_true=3,
    n_flagged_false=3,
    force_dino=False,
    seed=1
)

print("\n✅ Tüm qualitative figure'lar oluşturuldu!")
print(f"📁 Çıktı dizini: {os.path.join(OUT_ROOT, 'qual_panels_balanced')}")